# 29_invariants: What Survives Every Transformation?

**Lab report:** Invariant Specification Demystifies Translation

The previous notebooks studied increasingly strong transformations:

| Notebook | Transformation | Core question |
|---|---|---|
| 13 | Translation | What survives language-to-language mapping? |
| 17 | Script change | What survives visual re-representation? |
| 23 | Language evolution | What survives descendant-language derivation? |

Notebook 29 asks the synthesis question:

> Which invariant best explains all three transformations?

## 1. Setup

Reuse the upstream GLOSSOPETRAE repo and any CSV/JSON outputs produced by earlier notebooks.

In [ ]:
from pathlib import Path
import json
import csv
import os
import subprocess
from collections import defaultdict

ROOT = Path.cwd()
UPSTREAM_URL = "https://github.com/elder-plinius/GLOSSOPETRAE.git"
UPSTREAM_DIR = ROOT / "GLOSSOPETRAE"

if not UPSTREAM_DIR.exists():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)
else:
    print("Upstream repo already present:", UPSTREAM_DIR)

FIGURES = ROOT / "figures"
DATA = ROOT / "data"
FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

print("Repository exists:", UPSTREAM_DIR.exists())
print("Data files:", [p.name for p in DATA.glob("*")][:20])

## 2. Candidate invariants

The candidate invariants are deliberately broad. The point is to compare them across transformations, not assume the answer.

In [ ]:
candidate_invariants = [
    {
        "candidate": "words",
        "definition": "Surface lexical forms or written word strings.",
        "prediction": "Preserved under script transliteration sometimes; weak under translation and evolution."
    },
    {
        "candidate": "phonology",
        "definition": "Sound inventory, pronunciation, sound constraints.",
        "prediction": "Often changes under translation and evolution; may persist under script change."
    },
    {
        "candidate": "glyphs",
        "definition": "Visual symbols or SVG writing forms.",
        "prediction": "Changes under script change; weak general invariant."
    },
    {
        "candidate": "grammar",
        "definition": "Syntax, morphology, compositional form.",
        "prediction": "Often partially preserved; stronger than words but weaker than correspondence."
    },
    {
        "candidate": "meaning",
        "definition": "Conceptual content or intended proposition.",
        "prediction": "Strong under translation and script change; may drift under evolution."
    },
    {
        "candidate": "lineage",
        "definition": "Ancestor-descendant relationship through time.",
        "prediction": "Central for evolution; not central for direct translation or script change."
    },
    {
        "candidate": "correspondence",
        "definition": "Mapped relationships among forms, meanings, scripts, and descendants.",
        "prediction": "Expected to survive all three transformations when the system tracks mappings."
    },
    {
        "candidate": "specification",
        "definition": "The constraint set that generates language, script, translation, and descendants.",
        "prediction": "Strongest if the engine exposes a stable seed/config/object across transformations."
    },
]

try:
    import pandas as pd
    cand_df = pd.DataFrame(candidate_invariants)
    display(cand_df)
    cand_df.to_csv(DATA / "29_candidate_invariants.csv", index=False)
except Exception:
    for row in candidate_invariants:
        print(row)

## 3. Transformation scorecard

Score each candidate across the three transformations.

Use:

- `1.0` = preserved or directly supported,
- `0.5` = partially preserved or transformation-dependent,
- `0.0` = changes or not preserved,
- `None` = not applicable or not observed.

The initial scores are hypotheses. Update them after inspecting actual upstream outputs.

In [ ]:
score_rows = [
    # candidate, translation, script_change, evolution
    {"candidate": "words", "translation": 0.0, "script_change": 0.5, "evolution": 0.0,
     "notes": "Surface words are not stable across translation or long-term change."},
    {"candidate": "phonology", "translation": 0.0, "script_change": 0.75, "evolution": 0.5,
     "notes": "Pronunciation may remain stable under script change; sound shifts during evolution."},
    {"candidate": "glyphs", "translation": 0.0, "script_change": 0.0, "evolution": 0.0,
     "notes": "Glyphs are representation-layer forms and are expected to change."},
    {"candidate": "grammar", "translation": 0.5, "script_change": 1.0, "evolution": 0.5,
     "notes": "Grammar is more stable than words but may differ across languages and descendants."},
    {"candidate": "meaning", "translation": 1.0, "script_change": 1.0, "evolution": 0.5,
     "notes": "Meaning is strong for translation and script change; evolution may introduce drift."},
    {"candidate": "lineage", "translation": None, "script_change": None, "evolution": 1.0,
     "notes": "Lineage is mainly an evolution invariant."},
    {"candidate": "correspondence", "translation": 1.0, "script_change": 1.0, "evolution": 1.0,
     "notes": "Correspondence can relate changed forms across all transformations."},
    {"candidate": "specification", "translation": 1.0, "script_change": 1.0, "evolution": 1.0,
     "notes": "Specification is the proposed generative invariant if exposed by the engine."},
]

try:
    import pandas as pd
    score_df = pd.DataFrame(score_rows)
    numeric_cols = ["translation", "script_change", "evolution"]
    score_df["mean_score"] = score_df[numeric_cols].mean(axis=1, skipna=True)
    display(score_df.sort_values("mean_score", ascending=False))
    score_df.to_csv(DATA / "29_invariant_scorecard.csv", index=False)
except Exception:
    for row in score_rows:
        print(row)

## 4. Evidence from earlier notebooks

Load available outputs from notebooks 13, 17, and 23 if they exist.

This notebook is designed to work even if some files are missing.

In [ ]:
import json
from pathlib import Path

def load_json_if_exists(path):
    path = Path(path)
    if path.exists():
        try:
            return json.loads(path.read_text())
        except Exception as e:
            return {"error": str(e)}
    return None

evidence_files = {
    "translation_scores": DATA / "13_invariant_candidate_scores.csv",
    "script_scores": DATA / "17_script_change_invariants.csv",
    "descendant_scores": DATA / "23_v2_invariant_scores.csv",
    "family_run": DATA / "23_v2_real_family_run_seed42.json",
    "family_summary": DATA / "23_v2_real_run_summary.csv",
    "family_structure": DATA / "23_v2_output_structure.csv",
}

evidence_presence = []
for label, path in evidence_files.items():
    evidence_presence.append({
        "label": label,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
    })

try:
    import pandas as pd
    evidence_df = pd.DataFrame(evidence_presence)
    display(evidence_df)
    evidence_df.to_csv(DATA / "29_evidence_presence.csv", index=False)
except Exception:
    for row in evidence_presence:
        print(row)

family_run = load_json_if_exists(DATA / "23_v2_real_family_run_seed42.json")
print("family_run loaded:", family_run is not None)

## 5. Metadata search inside the real family run

If Notebook 23 produced a JSON run, search it for invariant-relevant terms.

In [ ]:
def contains_key_like(obj, terms):
    if isinstance(obj, dict):
        for k, v in obj.items():
            kl = str(k).lower()
            if any(t in kl for t in terms):
                return True
            if contains_key_like(v, terms):
                return True
    elif isinstance(obj, list):
        return any(contains_key_like(x, terms) for x in obj)
    return False

term_groups = {
    "words": ["word", "lex", "vocab", "root"],
    "phonology": ["phon", "sound", "syllable", "vowel", "consonant"],
    "glyphs": ["glyph", "script", "orthograph", "writing"],
    "grammar": ["grammar", "syntax", "morph"],
    "meaning": ["meaning", "concept", "semantic", "gloss"],
    "lineage": ["lineage", "ancestor", "proto", "family", "daughter", "descendant"],
    "correspondence": ["correspond", "mapping", "translate", "translation"],
    "specification": ["spec", "seed", "constraint", "config"],
}

metadata_rows = []
if family_run is not None:
    for candidate, terms in term_groups.items():
        metadata_rows.append({
            "candidate": candidate,
            "observed_in_family_run": contains_key_like(family_run, terms),
            "terms": ", ".join(terms),
        })
else:
    for candidate, terms in term_groups.items():
        metadata_rows.append({
            "candidate": candidate,
            "observed_in_family_run": None,
            "terms": ", ".join(terms),
        })

try:
    import pandas as pd
    meta_df = pd.DataFrame(metadata_rows)
    display(meta_df)
    meta_df.to_csv(DATA / "29_family_run_metadata_search.csv", index=False)
except Exception:
    for row in metadata_rows:
        print(row)

## 6. Invariant comparison figure

This plot compares the candidate invariants across translation, script change, and evolution.

Higher bars mean the candidate explains more transformations.

In [ ]:
import matplotlib.pyplot as plt

try:
    plot_df = score_df.copy()
except NameError:
    import pandas as pd
    plot_df = pd.DataFrame(score_rows)
    plot_df["mean_score"] = plot_df[["translation", "script_change", "evolution"]].mean(axis=1, skipna=True)

plot_df = plot_df.sort_values("mean_score", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot_df["candidate"], plot_df["mean_score"])
ax.set_xlabel("Mean preservation score")
ax.set_title("Candidate invariants across translation, script change, and evolution")
ax.set_xlim(0, 1.05)

path = FIGURES / "29_invariant_scorecard.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
print("Saved:", path)
plt.show()

## 7. Transformation lattice figure

This figure summarizes the repo-level argument.

The same upstream system supports three transformations:

```text
translation
script change
language evolution
```

The strongest shared explanation is an invariant specification or correspondence structure.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
ax.axis("off")

nodes = {
    "Specification\\n(seed + constraints)": (0.50, 0.90),
    "Language\\n(grammar + lexicon + sound)": (0.50, 0.72),
    "Translation": (0.22, 0.50),
    "Script Change": (0.50, 0.50),
    "Evolution": (0.78, 0.50),
    "Language B": (0.22, 0.30),
    "Script B": (0.50, 0.30),
    "Descendants": (0.78, 0.30),
    "Shared Invariant\\ncorrespondence / specification": (0.50, 0.10),
}

for label, (x, y) in nodes.items():
    ax.text(
        x, y, label,
        ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.45", linewidth=1.2),
        fontsize=10
    )

edges = [
    ("Specification\\n(seed + constraints)", "Language\\n(grammar + lexicon + sound)"),
    ("Language\\n(grammar + lexicon + sound)", "Translation"),
    ("Language\\n(grammar + lexicon + sound)", "Script Change"),
    ("Language\\n(grammar + lexicon + sound)", "Evolution"),
    ("Translation", "Language B"),
    ("Script Change", "Script B"),
    ("Evolution", "Descendants"),
    ("Language B", "Shared Invariant\\ncorrespondence / specification"),
    ("Script B", "Shared Invariant\\ncorrespondence / specification"),
    ("Descendants", "Shared Invariant\\ncorrespondence / specification"),
]

for a, b in edges:
    x1, y1 = nodes[a]
    x2, y2 = nodes[b]
    ax.annotate(
        "",
        xy=(x2, y2 + 0.04),
        xytext=(x1, y1 - 0.04),
        arrowprops=dict(arrowstyle="->", linewidth=1)
    )

path = FIGURES / "29_transformation_lattice.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
print("Saved:", path)
plt.show()

## 8. Interpretation

The scorecard suggests:

- **Words** are too surface-level.
- **Glyphs** are representation-level and change easily.
- **Phonology** is stable only under some transformations.
- **Grammar** and **meaning** are stronger, but may drift under evolution.
- **Lineage** is powerful for descendants but not general across translation or script change.
- **Correspondence** and **specification** are the strongest cross-transformation candidates.

This supports the lab-report title:

> Invariant Specification Demystifies Translation

with a refinement:

> Correspondence is the measurable face of specification.

## 9. Engineering statement

**Result**

GLOSSOPETRAE makes language transformations inspectable by exposing a generative specification layer.

Translation, script change, and language evolution alter observable forms.

The strongest invariant candidates are not words, glyphs, or pronunciations.

The strongest candidates are:

- specification,
- correspondence,
- lineage where evolution is involved.

**Statement**

Languages evolve. Representations change. Correspondence persists when transformations are generated from a shared specification.

## 10. Next work

A follow-up notebook could move from structural inspection to quantitative tests:

| Notebook | Focus |
|---|---|
| 31 | Extract comparable lexicons from generated families |
| 37 | Measure descendant similarity over different century values |
| 43 | Compare multiple seeds |
| 49 | Test whether correspondence degrades smoothly with time |
| 53 | Build report-ready figures from actual generated outputs |